In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson

# Boilerplate

In [ ]:
#[1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def bandpower(data, sf, band, window_sec=None, relative=False):
    """Compute the average power of the signal x in a specific frequency band.

    Parameters
    ----------
    data : 1d-array
        Input signal in the time-domain.
    sf : float
        Sampling frequency of the data.
    band : list
        Lower and upper frequencies of the band of interest.
    window_sec : float
        Length of each window in seconds.
        If None, window_sec = (1 / min(band)) * 2
    relative : boolean
        If True, return the relative power (= divided by the total power of the signal).
        If False (default), return the absolute power.

    Return
    ------
    bp : float
        Absolute or relative band power.
    """

    band = np.asarray(band)
    low, high = band

    psd, freqs = mne.time_frequency.psd_array_multitaper(data, sf, fmin=low, fmax=high, adaptive=True, low_bias=True, normalization='full', verbose=False)
    

    # Frequency resolution
    #freq_res = freqs[1] - freqs[0]
    difference = np.diff(freqs)
    freq_res = np.mean(difference)

    # Find closest indices of band in frequency vector
    idx_band = np.logical_and(freqs >= low, freqs <= high)

    # Integral approximation of the spectrum using Simpson's rule.
    integral = simpson(psd[idx_band], dx=freq_res)
    average = np.mean(psd[idx_band])

    if relative:
        bp /= simpson(psd, dx=freq_res)
    return integral, average

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
cfg = load_config()
cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")


freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

all_subject_power_data_integral = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
all_subject_power_data_average = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}

# preprocess data (get power per frequency band and amplitudes for each trial)




"""
#all_subject_amplitude_data = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
#all_subject_uncertainty_data = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.data_directory = dir_path
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]

    #pred_label = np.zeros((all_epochs.shape[0]))
    #uncertainties = np.zeros((all_epochs.shape[0]))                
    #for i in tqdm(range(0, len(all_epochs))):
    #    start_index = i+100
    #    inputs = torch.from_numpy(all_epochs[i])
    #    inputs = inputs.to(device).float()
    #    inputs = inputs.unsqueeze(0)
        # construct function to load correct model for current trial
    #    model = load_model(start_index=start_index, subject_index=subject_index)
    #    pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
    #    var = torch.exp(log_var)
    #    pred_label[i] = pred_mean.cpu().detach().numpy()
    #    uncertainties[i] = var.cpu().detach().numpy()
    #all_subject_amplitude_data[subject_index] = pred_label
    #all_subject_uncertainty_data[subject_index] = uncertainties

    #print(all_epochs.shape)
    all_trials_integral = []
    all_trials_average = []
    for trial in range(all_epochs.shape[0]):
        all_data_trial_integral = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}
        all_data_trial_average = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}

        for idx,ch in enumerate(ch_names):
            #print(ch)
            data = all_epochs[trial, idx, :]
            for band_name, band_range in freq_bands.items():
                integral,average = bandpower(data, 1000, band_range)
                all_data_trial_integral[band_name][ch].append(integral)
                all_data_trial_average[band_name][ch].append(average)

        all_trials_integral.append(all_data_trial_integral)
        all_trials_average.append(all_data_trial_average)
    all_subject_power_data_integral[subject_index] = all_trials_integral
    all_subject_power_data_average[subject_index] = all_trials_average
    os.makedirs(dir_path, exist_ok=True)
    np.save(os.path.join(dir_path, "all_subject_power_data_integral.npy"), all_subject_power_data_integral)
    np.save(os.path.join(dir_path, "all_subject_power_data_average.npy"), all_subject_power_data_average)
"""

def process_subject_data(cfg, subject_index, dir_path, freq_bands, device):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index

    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]

    all_trials_integral = []
    all_trials_average = []
    for trial in range(all_epochs.shape[0]):
        all_data_trial_integral = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}
        all_data_trial_average = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}

        for idx, ch in enumerate(ch_names):def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/finetune_model_weights"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_pretrain_subject_index_{subject_index}_start_idx_{start_index}.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model
                integral, average = bandpower(data, 1000, band_range)
                all_data_trial_integral[band_name][ch].append(integral)
                all_data_trial_average[band_name][ch].append(average)


        all_trials_integral.append(all_data_trial_integral)
        all_trials_average.append(all_data_trial_average)
        
    return all_trials_integral, all_trials_average

#for subject_index in cfg.dataset.test_subject_indices:
    all_trials_integral, all_trials_average = process_subject_data(cfg, subject_index, dir_path, freq_bands, device)
    all_subject_power_data_integral[subject_index] = all_trials_integral
    all_subject_power_data_average[subject_index] = all_trials_average

#os.makedirs(dir_path, exist_ok=True)
#np.save(os.path.join(dir_path, "all_subject_power_data_integral.npy"), all_subject_power_data_integral)
#np.save(os.path.join(dir_path, "all_subject_power_data_average.npy"), all_subject_power_data_average)
#Save the data


#np.save(os.path.join(dir_path, "all_subject_amplitude_data.npy"), all_subject_amplitude_data)
#np.save(os.path.join(dir_path, "all_subject_uncertainty_data.npy"), all_subject_uncertainty_data)
print("Data saved")


subject_ x channel x freq_band

save data after run!

In [ ]:
import pickle
#cfg = load_config()

all_subject_power_data_integral = np.load(os.path.join(dir_path, "all_subject_power_data_integral.npy"), allow_pickle=True).item()
all_subject_power_data_average = np.load(os.path.join(dir_path, "all_subject_power_data_average.npy"), allow_pickle=True).item()
#all_subject_amplitude_data = np.load(os.path.join(dir_path, "all_subject_amplitude_data.npy"), allow_pickle=True).item()
#all_subject_uncertainty_data = np.load(os.path.join(dir_path, "all_subject_uncertainty_data.npy"), allow_pickle=True).item()
all_subject_amplitude_data = {}
all_subject_uncertainty_data = {}
data_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject in cfg.dataset.test_subject_indices:
    with open(os.path.join(data_path, f"subject_{subject}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        all_subject_amplitude_data[subject] = data['predictions']
        all_subject_uncertainty_data[subject] = data['uncertainties']
        



# subject 2

## compare power per frequency band vs predicted amplitude

In [ ]:
def plot_power_amplitude(all_subject_power_data, all_subject_amplitude_data,subject_index=2, show_plot=True):
    all_trials = all_subject_power_data[subject_index]
    all_amplitudes = all_subject_amplitude_data[subject_index]
    all_uncertainties = all_subject_uncertainty_data[subject_index]
    # data is of the shape subject_index, trial, band, channel
    # make separate plot for each channel and separate subplot for each frequency band where power is on the x-axis and amplitude is on the y-axis
    
    freq_bands = {"theta": (4, 8),
                    "alpha": (8, 12),
                    "beta": (12, 30),
                    "gamma": (30, 45)}
    ch_names = list(all_trials[0]["theta"].keys())
    high_correlations = {f: {} for f in freq_bands.keys()}
    corrs_abs = {f: {} for f in freq_bands.keys()}
    corrs = {f: {} for f in freq_bands.keys()}

    for ch_idx, ch in enumerate(ch_names):
        if show_plot:
            fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(10, 2), sharey=True)
            fig.suptitle(f"Channel: {ch}")
        for band_idx, band_name in enumerate(freq_bands.keys()):
            current_data = []
            for trial in range(len(all_trials)):
                power = all_trials[trial][band_name][ch]
                current_data.append(power[0])
            # throw away trials with top 1% of power and corresponding amplitudes
            current_data = np.array(current_data)
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr = np.corrcoef(current_data[indices], all_amplitudes[indices])[0,1]
            corrs_abs[band_name][ch] = np.abs(corr)
            corrs[band_name][ch] = corr

            if show_plot:            
            
                axs[band_idx].set_xlabel('Power')
                axs[band_idx].set_ylabel('Amplitude')
                if np.abs(corr) >= 0.3:
                    axs[band_idx].set_title(f"{band_name}, corr: {corr:.2f}")
                    high_correlations[band_name][ch] = corr
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')

                else:
                    axs[band_idx].set_title(f"{band_name}")
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5)
                fig.savefig(f"correlation_plots/channel_{ch}_subject_{subject_index}.png")
                plt.show()
    return high_correlations, corrs, corrs_abs

In [ ]:
def plot_power_amplitude_filtered(all_subject_power_data, all_subject_amplitude_data, subject_index=2, treshhold=0.3, show_plot=True, index_groups=None):
    all_trials = all_subject_power_data[subject_index]
    all_amplitudes = all_subject_amplitude_data[subject_index]
    all_uncertainties = all_subject_uncertainty_data[subject_index]
    
    if index_groups is not None:
        all_amplitudes = all_amplitudes[index_groups]
        all_trials = [trial for i, trial in enumerate(all_trials) if index_groups[i]]
    
    freq_bands = {"theta": (4, 8),
                  "alpha": (8, 12),
                  "beta": (12, 30),
                  "gamma": (30, 45)}
    ch_names = list(all_trials[0]["theta"].keys())
    high_correlations = {f: {} for f in freq_bands.keys()}
    
    for ch in ch_names:
        for band_idx, band_name in enumerate(freq_bands.keys()):
            current_data = []
            for trial in range(len(all_trials)):
                power = all_trials[trial][band_name][ch]
                current_data.append(power[0])
            
            current_data = np.array(current_data)
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr = np.corrcoef(current_data[indices], all_amplitudes[indices])[0, 1]
            
            if treshhold == -1 or np.abs(corr) >= treshhold:
                high_correlations[band_name][ch] = corr
                if show_plot:
                    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 3))
                    fig.suptitle(f"Subject {subject_index}, Channel: {ch}")
                    ax.set_xlabel('Power')
                    ax.set_ylabel('predicted amplitude')
                    ax.set_title(f"{band_name}, corr: {corr:.2f}", y=0.95)
                    ax.scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')
                    os.makedirs("correlation_plots", exist_ok=True)
                    fig.savefig(f"correlation_plots/subject_{subject_index}_channel_{ch}_band_{band_name}.png")
        
    return high_correlations


In [ ]:
def get_fixed_median(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    return fixed_median, ch_names

In [ ]:
pred_binary_fixed = {}
for subject_index in cfg.dataset.test_subject_indices:
    fixed_median, ch_names = get_fixed_median(subject_index=subject_index)
    pred_binary_fixed[subject_index] = np.where(all_subject_amplitude_data[subject_index] > fixed_median, 1, 0)

In [ ]:
groupby_dicts_list = {}
for subject_index in cfg.dataset.test_subject_indices:
    if subject_index  == 25:
        continue
    remove_indices = []
    sigma2 = all_subject_uncertainty_data[subject_index ]
    # throw away top 5% of values, hoping to remove outliers
    sigma2_indices = sigma2<=np.percentile(sigma2, 95)
    outlier_indices = ~sigma2_indices
    #hist_values.append(sigma2[i][vars_lower_x_percent_bool])   

    #This way the treshholds are calculated without the outliers
    low_threshold = np.percentile(sigma2[sigma2_indices], 33)
    high_threshold = np.percentile(sigma2[sigma2_indices], 66)
    # Treshholds are now defined without consideration of outliers.
    # BUT: outlier trials not removed from dataset!

    # Remove outliers by getting boolean array of them and using xor(^) with group_indices
    low_indices = (sigma2 < low_threshold)
    medium_indices = ((low_threshold<=sigma2) & (sigma2<=high_threshold))
    high_indices = (sigma2>high_threshold)^outlier_indices
    uncertainty_levels = {"low":low_indices, "medium":medium_indices, "high":high_indices}


    binary_label_fixed_zero_bool = pred_binary_fixed[subject_index] == 0
    binary_label_fixed_one_bool = pred_binary_fixed[subject_index] ==  1
    binary_label_fixed_dict = {"pred_zero" : binary_label_fixed_zero_bool, "pred_one" : binary_label_fixed_one_bool}

    
    groupby_dicts_list[subject_index] = binary_label_fixed_dict

In [ ]:
index_groups_all = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    index_groups_subject = []
    start = 0
    end = 0
    while end < len(all_subject_uncertainty_data[subject_index]):
        if len(all_subject_uncertainty_data[subject_index]) >= end + 100:
            end += 100
        else:
            end = len(all_subject_uncertainty_data[subject_index])

        index_group = np.zeros(len(all_subject_uncertainty_data[subject_index]), dtype=bool)
        index_group[start:end] = True
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        start += 100
    index_groups_all[subject_index] = np.array(index_groups_subject)


In [ ]:
index_groups_all[1].shape

In [ ]:
#plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=2)

In [ ]:
plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=2, index_groups=index_groups_all[2][0])

In [ ]:
plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=2, index_groups=index_groups_all[2][3])

In [ ]:
#plot_power_amplitude(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=2)

In [ ]:
zz

## get raw labels for every subject

In [ ]:

def get_raw_labels(subject_index):
    cfg = load_config()
    cwd = os.getcwd()
    dir_path = os.path.join(cwd, "frequency_power_data")
    cfg = load_config()
    cfg.dataset.data_directory = dir_path
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    return all_labels_raw[150:]
  

In [ ]:
#cfg = load_config()
#all_subjects_raw_labels = {}
#for subject_index in cfg.dataset.test_subject_indices:
#    all_subjects_raw_labels[subject_index] = get_raw_labels(subject_index)

## compare power vs actual amplitude

In [ ]:

#plot_power_amplitude(all_subject_power_data_integral, all_subjects_raw_labels, subject_index=2)

# For all subjects

## compare power vs predicted labels

In [ ]:
all_subjects_highest_corrs = {}
for subject_index in cfg.dataset.test_subject_indices:
    all_subjects_highest_corrs[subject_index] = []
    for index_group in index_groups_all[subject_index]:
        all_subjects_highest_corrs[subject_index].append(plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, index_groups=index_group, show_plot=False))
    #all_subjects_highest_corrs = {subject_index: plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index) for subject_index in cfg.dataset.test_subject_indices}


In [ ]:
cfg = load_config()
all_subjects_highest_corrs_all = {subject_index: plot_power_amplitude(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)[1] for subject_index in cfg.dataset.test_subject_indices}
all_subjects_highest_corrs_all_abs = {subject_index: plot_power_amplitude(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)[2] for subject_index in cfg.dataset.test_subject_indices}



In [ ]:
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_total.npy"), all_subjects_highest_corrs_all)
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_total_abs.npy"), all_subjects_highest_corrs_all_abs)

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index)

# check correlation between 10 most important channels pe subject and power amplitude correlations

## top channel functions

In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}






In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

#Dont execute!
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.data_directory = dir_path
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]
    gradshap = compute_gradshap(all_epochs, subject_index=subject_index)
    np.save(os.path.join(dir_path, f"gradshap_subject_{subject_index}.npy"), gradshap)



## top 10 channels per subject per timepoint

In [ ]:
cfg = load_config()
top_10_per_subject = {}
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    top_10_per_subject[subject_index] = []
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    
    # Iterate over the inner list of all_subject_index for each subject
    for index_group in index_groups_all[subject_index]:
        gradshap_indexed = gradshap[index_group]
        print(gradshap_indexed.shape)
        top_10_channels, channel_point_dicts_individual_trials, sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap_indexed)}, "gradshap", ch_names, 10, update_channel_points_linear)
        top_10_per_subject[subject_index].append(top_10_channels)

## top 60 channels per subject

In [ ]:
cfg = load_config()
top_60_per_subject = {}
top_60_per_subject_dict = {}

for subject_index in cfg.dataset.test_subject_indices:
    top_60_per_subject[subject_index] = []
    top_60_per_subject_dict[subject_index] = []

    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    
    for index_group in index_groups_all[subject_index]:
        gradshap_indexed = gradshap[index_group]
        top_k_channels, channel_point_dicts_individual_trials, top_k_channels_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap_indexed)}, "gradshap", ch_names, 60, update_channel_points_linear)
        top_60_per_subject[subject_index].append(top_k_channels)
        top_60_per_subject_dict[subject_index].append(top_k_channels_dict)

In [ ]:
top_60_per_subject_dict = {subject_index: top_60_per_subject_dict[i] for i, subject_index in enumerate(cfg.dataset.test_subject_indices)}

In [ ]:
all_subjects_highest_corrs[2]

## for gamma channel (most relevant)

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, frequency_band='gamma'):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        for i in range(len(index_groups_all[subject_index])):
            #print(f"Subject {subject_index}:")
            #print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
            #print(f"Channels with high correlation in {frequency_band} band:", list(all_subjects_highest_corrs[subject_index][frequency_band].keys()))

            common_channels = set(top_k_per_subject[subject_index][i]) & set(all_subjects_highest_corrs[subject_index][i][frequency_band].keys())
            num_common_channels = len(common_channels)
            num_significant_channels = len(all_subjects_highest_corrs[subject_index][i][frequency_band].keys())
            denominator = max(1, min(len(top_k_per_subject[subject_index][i]), len(all_subjects_highest_corrs[subject_index][i][frequency_band].keys())))
            ratio = num_common_channels / denominator
            subject_common_channels.append(num_common_channels)
            subject_ratios.append(ratio)
            subject_sig_channels.append(num_significant_channels)
            print(f"Number of common channels: {num_common_channels}")
            print(f"Ratio: {ratio:.2f}")
            print()

    bar_width = 0.35   
    colors = plt.cm.viridis(np.linspace(0, 1, 8))

    for i, subject_index in enumerate(cfg.dataset.test_subject_indices):
        fig, ax = plt.subplots(figsize=(14, 6))
        for j in range(len(index_groups_all[subject_index])):
            ax.bar(j, subject_ratios[i * len(index_groups_all[subject_index]) + j], bar_width, color=colors[j], label=f'Group {j}')

        ax.set_xlabel('Group Index')
        ax.set_ylabel('Ratio of Common Channels')
        ax.set_title(f'Common Channels Ratio for {frequency_band} Band - Subject {subject_index}')
        ax.set_xticks(np.arange(len(index_groups_all[subject_index])))
        ax.set_xticklabels([f"Group {j}" for j in range(len(index_groups_all[subject_index]))], rotation=45)
        ax.legend()

        plt.tight_layout()
        plt.savefig(f"common_channels_ratio_{frequency_band}_subject_{subject_index}.png")
        plt.show()

# Example usage:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='gamma')



In [ ]:
top_10_per_subject

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, frequency_band='gamma'):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band:", list(all_subjects_highest_corrs[subject_index][frequency_band].keys()))

        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index][frequency_band].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"Number of common channels: {num_common_channels}")
        print(f"Ratio: {ratio:.2f}")
        print()
    
    empty_list_indices = subject_common_channels[subject_common_channels ==0]
    fig = plt.figure(figsize=(10, 4))
    plt.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            plt.plot(idx, subject_ratios[idx], 'ro')

    plt.xticks(np.arange(len(subject_ratios)), cfg.dataset.test_subject_indices, rotation=45)
    plt.xlabel('Subject Index')
    plt.ylabel('Ratio of Common Channels')
    plt.title(f'Common Channels Ratio for {frequency_band} Band')
    plt.savefig(f"common_channels_ratio_{frequency_band}.png")
    plt.show()

# Example usage:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='gamma')

In [ ]:
all_subjects_highest_corrs

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, frequency_band='gamma'):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band:", list(all_subjects_highest_corrs[subject_index][frequency_band].keys()))

        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index][frequency_band].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"Number of common channels: {num_common_channels}")
        print(f"Ratio: {ratio:.2f}")
        print()
    
    empty_list_indices = subject_common_channels[subject_common_channels ==0]
    fig = plt.figure(figsize=(10, 4))
    plt.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            plt.plot(idx, subject_ratios[idx], 'ro')

    plt.xticks(np.arange(len(subject_ratios)), cfg.dataset.test_subject_indices, rotation=45)
    plt.xlabel('Subject Index')
    plt.ylabel('Ratio of Common Channels')
    plt.title(f'Common Channels Ratio for {frequency_band} Band')
    plt.savefig(f"common_channels_ratio_{frequency_band}.png")
    plt.show()

# Example usage:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='gamma')



## for beta channel

In [ ]:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='beta')

## for alpha channel

In [ ]:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='alpha')

## for theta channel

In [ ]:
compare_channels(cfg, top_10_per_subject, all_subjects_highest_corrs, frequency_band='theta')

there definitely seems to be some correlation between the gamma band power, and the predicted amplitude for some channels&subjects as well as a correlation between power x amplitude correlation and most important channels

get top k-channels and check if there is a correlation with power amplitude correlation 
Potentially also check relative power

In [ ]:
#all_subjects_corrs = {subject_index: plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False, treshhold=-1) for subject_index in cfg.dataset.test_subject_indices}
all_subject_corrs = {}
for subject_index in cfg.dataset.test_subject_indices:
    all_subject_corrs[subject_index] = []
    for i in range(len(index_groups_all[subject_index])):
        all_subject_corrs[subject_index].append(plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, index_groups=index_groups_all[subject_index][i], show_plot=False, treshhold=-1))

In [ ]:
all_subject_corrs[1][0]

In [ ]:
def extract_top_10_channels_and_mean_corr(all_subjects_corrs):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, freq_bands in all_subjects_corrs.items():
        top_10_channels_per_subject[subject] = {}
        mean_corr_per_subject[subject] = {}

        for index_group in range(len(freq_bands)):
            top_10_channels_per_subject[subject][index_group] = {}
            mean_corr_per_subject[subject][index_group] = {}
            for band, channels in freq_bands[index_group].items():
                sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:10]
                top_10_channels_per_subject[subject][index_group][band] = dict(sorted_channels)

                mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
                mean_corr_per_subject[subject][index_group][band] = mean_corr
    return top_10_channels_per_subject, mean_corr_per_subject


top_10_channels_per_subject, mean_corr_per_subject = extract_top_10_channels_and_mean_corr(all_subject_corrs)

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='gamma')

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='alpha')

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='beta')

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='theta')

# check across frequency bands

In [ ]:
all_subjects_corrs_max = {}

for subject_index in cfg.dataset.test_subject_indices:
    all_subjects_corrs_max[subject_index] = {}
    for channel in ch_names:
        max_corr = 0
        for band in freq_bands.keys():
            corr = abs(all_subjects_corrs[subject_index][band].get(channel, 0))
            if corr > max_corr:
                max_corr = corr
        all_subjects_corrs_max[subject_index][channel] = max_corr

In [ ]:
def extract_top_10_channels_and_mean_corr_all_freqband(all_subjects_corrs):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, channels in all_subjects_corrs.items():
        sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:10]
        top_10_channels_per_subject[subject] = dict(sorted_channels)
        
        mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
        mean_corr_per_subject[subject] = mean_corr
    
    return top_10_channels_per_subject, mean_corr_per_subject

top_10_channels_per_subject, mean_corr_per_subject = extract_top_10_channels_and_mean_corr_all_freqband(all_subjects_corrs_max)


In [ ]:
def compare_channels_all_freqband(cfg, top_k_per_subject, all_subjects_highest_corrs):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation:", list(all_subjects_highest_corrs[subject_index].keys()))

        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"Number of common channels: {num_common_channels}")
        print(f"Ratio: {ratio:.2f}")
        print()
    
    fig = plt.figure(figsize=(10, 4))
    plt.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            plt.plot(idx, subject_ratios[idx], 'ro')

    plt.xticks(np.arange(len(subject_ratios)), cfg.dataset.test_subject_indices, rotation=45)
    plt.xlabel('Subject Index')
    plt.ylabel('Ratio of Common Channels')
    plt.title('Common Channels Ratio')
    plt.savefig("common_channels_ratio.png")
    plt.show()

# Example usage:
compare_channels_all_freqband(cfg, top_10_per_subject, 
top_10_channels_per_subject)



# compare absolute and relative power results

In [ ]:
def compare_channels2(cfg, top_k_per_subject, all_subjects_highest_corrs_1, all_subjects_highest_corrs_2, frequency_band='gamma'):
    subject_ratios_1 = []
    subject_ratios_2 = []
    subject_common_channels_1 = []
    subject_common_channels_2 = []
    subject_sig_channels_1 = []
    subject_sig_channels_2 = []
    subject_agreement = []

    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band (set 1):", list(all_subjects_highest_corrs_1[subject_index][frequency_band].keys()))
        print(f"Channels with high correlation in {frequency_band} band (set 2):", list(all_subjects_highest_corrs_2[subject_index][frequency_band].keys()))

        common_channels_1 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        common_channels_2 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())

        num_common_channels_1 = len(common_channels_1)
        num_common_channels_2 = len(common_channels_2)

        num_significant_channels_1 = len(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        num_significant_channels_2 = len(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())

        denominator_1 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_1))
        denominator_2 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_2))

        ratio_1 = num_common_channels_1 / denominator_1
        ratio_2 = num_common_channels_2 / denominator_2

        subject_common_channels_1.append(num_common_channels_1)
        subject_common_channels_2.append(num_common_channels_2)

        subject_ratios_1.append(ratio_1)
        subject_ratios_2.append(ratio_2)

        subject_sig_channels_1.append(num_significant_channels_1)
        subject_sig_channels_2.append(num_significant_channels_2)

        top_10_set_1 = set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        top_10_set_2 = set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
        agreement = len(top_10_set_1 & top_10_set_2) / 10
        subject_agreement.append(agreement)

        print(f"Number of common channels absolute power: {num_common_channels_1}")
        print(f"Ratio absolute power: {ratio_1:.2f}")
        print(f"Number of common channels relative power: {num_common_channels_2}")
        print(f"Ratio relative power: {ratio_2:.2f}")
        print(f"Agreement between top 10 channels: {agreement:.2f}")
        print()

    mean_ratio_1 = np.mean(subject_ratios_1)
    mean_ratio_2 = np.mean(subject_ratios_2)
    mean_agreement = np.mean(subject_agreement)

    fig, ax = plt.subplots(figsize=(10, 4))
    bar_width = 0.35
    index = np.arange(len(subject_ratios_1))

    bar1 = ax.bar(index, subject_ratios_1, bar_width, label='absolute power')
    bar2 = ax.bar(index + bar_width, subject_ratios_2, bar_width, label='relative power')

    ax.set_xlabel('Subject Index')
    ax.set_ylabel('Ratio of Common Channels')
    ax.set_title(f'Common Channels Ratio for {frequency_band} Band')
    ax.set_xticks(index + bar_width / 2)
    ax.set_xticklabels(cfg.dataset.test_subject_indices, rotation=45)
    ax.legend()

    plt.tight_layout()
    plt.savefig(f"common_channels_ratio_{frequency_band}.png")
    plt.show()

    return mean_ratio_1, mean_ratio_2, mean_agreement

# Example usage:
top_10_absolute_power = np.load("top_10_absolute_power.npy", allow_pickle=True).item()
top_10_relative_power = np.load("top_10_relative_power.npy", allow_pickle=True).item()
mean_ratio_1, mean_ratio_2, mean_agreement = compare_channels2(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, frequency_band='gamma')
print(f"Mean common channels ratio (absolute power): {mean_ratio_1:.2f}")
print(f"Mean common channels ratio (relative power): {mean_ratio_2:.2f}")
print(f"Mean agreement between top 10 channels: {mean_agreement:.2f}")




# compare absolute power to relative power and periodic power

In [ ]:
# Example usage:
def compare_channels3(cfg, top_k_per_subject, all_subjects_highest_corrs_1, all_subjects_highest_corrs_2, all_subjects_highest_corrs_3, frequency_band='gamma'):
    subject_ratios_1 = []
    subject_ratios_2 = []
    subject_ratios_3 = []
    subject_common_channels_1 = []
    subject_common_channels_2 = []
    subject_common_channels_3 = []
    subject_sig_channels_1 = []
    subject_sig_channels_2 = []
    subject_sig_channels_3 = []
    subject_agreement_12 = []
    subject_agreement_13 = []
    subject_agreement_23 = []

    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band (set 1):", list(all_subjects_highest_corrs_1[subject_index][frequency_band].keys()))
        print(f"Channels with high correlation in {frequency_band} band (set 2):", list(all_subjects_highest_corrs_2[subject_index][frequency_band].keys()))
        print(f"Channels with high correlation in {frequency_band} band (set 3):", list(all_subjects_highest_corrs_3[subject_index][frequency_band].keys()))

        common_channels_1 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        common_channels_2 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
        common_channels_3 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

        num_common_channels_1 = len(common_channels_1)
        num_common_channels_2 = len(common_channels_2)
        num_common_channels_3 = len(common_channels_3)

        num_significant_channels_1 = len(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        num_significant_channels_2 = len(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
        num_significant_channels_3 = len(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

        denominator_1 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_1))
        denominator_2 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_2))
        denominator_3 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_3))

        ratio_1 = num_common_channels_1 / denominator_1
        ratio_2 = num_common_channels_2 / denominator_2
        ratio_3 = num_common_channels_3 / denominator_3

        subject_common_channels_1.append(num_common_channels_1)
        subject_common_channels_2.append(num_common_channels_2)
        subject_common_channels_3.append(num_common_channels_3)

        subject_ratios_1.append(ratio_1)
        subject_ratios_2.append(ratio_2)
        subject_ratios_3.append(ratio_3)

        subject_sig_channels_1.append(num_significant_channels_1)
        subject_sig_channels_2.append(num_significant_channels_2)
        subject_sig_channels_3.append(num_significant_channels_3)

        top_10_set_1 = set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
        top_10_set_2 = set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
        top_10_set_3 = set(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

        agreement_12 = len(top_10_set_1 & top_10_set_2) / 10
        agreement_13 = len(top_10_set_1 & top_10_set_3) / 10
        agreement_23 = len(top_10_set_2 & top_10_set_3) / 10

        subject_agreement_12.append(agreement_12)
        subject_agreement_13.append(agreement_13)
        subject_agreement_23.append(agreement_23)

        print(f"Number of common channels absolute power: {num_common_channels_1}")
        print(f"Ratio absolute power: {ratio_1:.2f}")
        print(f"Number of common channels relative power: {num_common_channels_2}")
        print(f"Ratio relative power: {ratio_2:.2f}")
        print(f"Number of common channels periodic power: {num_common_channels_3}")
        print(f"Ratio periodic power {ratio_3:.2f}")
        print(f"Agreement between absolute and relative power: {agreement_12:.2f}")
        print(f"Agreement between absolute and periodic component power {agreement_13:.2f}")
        print(f"Agreement between relative power periodic component power: {agreement_23:.2f}")
        print()

    mean_ratio_1 = np.mean(subject_ratios_1)
    mean_ratio_2 = np.mean(subject_ratios_2)
    mean_ratio_3 = np.mean(subject_ratios_3)
    std_ratio_1 = np.std(subject_ratios_1)
    std_ratio_2 = np.std(subject_ratios_2)
    std_ratio_3 = np.std(subject_ratios_3)
    mean_agreement_12 = np.mean(subject_agreement_12)
    mean_agreement_13 = np.mean(subject_agreement_13)
    mean_agreement_23 = np.mean(subject_agreement_23)

    fig, ax = plt.subplots(figsize=(10, 4))
    bar_width = 0.25
    index = np.arange(len(subject_ratios_1))

    bar1 = ax.bar(index, subject_ratios_1, bar_width, label='absolute power')
    bar2 = ax.bar(index + bar_width, subject_ratios_2, bar_width, label='relative power')
    bar3 = ax.bar(index + 2 * bar_width, subject_ratios_3, bar_width, label='periodic component power')

    ax.set_xlabel('Subject Index')
    ax.set_ylabel('Ratio of Common Channels')
    ax.set_title(f'Common Channels Ratio for {frequency_band} Band')
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(cfg.dataset.test_subject_indices, rotation=45)
    ax.legend()

    plt.tight_layout()
    plt.savefig(f"common_channels_ratio_{frequency_band}.png")
    plt.show()

    return mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23

# Example usage:
top_10_absolute_power = np.load("top_10_absolute_power.npy", allow_pickle=True).item()
top_10_relative_power = np.load("top_10_relative_power.npy", allow_pickle=True).item()
top_10_periodic_power = np.load("top_10_periodic_component.npy", allow_pickle=True).item()
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 = compare_channels3(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, top_10_periodic_power , frequency_band='gamma')




In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 = compare_channels3(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, top_10_periodic_power , frequency_band='alpha')

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 = compare_channels3(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, top_10_periodic_power , frequency_band='beta')

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 = compare_channels3(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, top_10_periodic_power , frequency_band='theta')

In [ ]:
mean_ratio_1 ,mean_ratio_2, mean_ratio_3, std_ratio_1, std_ratio_2, std_ratio_3, mean_agreement_12, mean_agreement_13, mean_agreement_23 

In [ ]:
def get_k_channels(ch_names, k=10):
    rng = np.random.default_rng()
    return rng.choice(ch_names, k, replace=False)
    

In [ ]:
def simulate_agreement(ch_names, k=10, n=1000000):
    agreement = np.zeros(k)
    for _ in tqdm(range(n)):
        top_k_channels = get_k_channels(ch_names, k)
        top_k_channels2 = get_k_channels(ch_names, k)
        intersection = set(top_k_channels) & set(top_k_channels2)
        agreement[len(intersection)] += 1
    
    return agreement

In [ ]:
agreement = simulate_agreement(ch_names, k=10, n=1000000)

In [ ]:
agreement/1000000

In [ ]:
np.cumsum(agreement/1000000)

In [ ]:
def compare_channels_all_bands(cfg, top_k_per_subject, all_subjects_highest_corrs_1, all_subjects_highest_corrs_2, all_subjects_highest_corrs_3):
    frequency_bands = ['theta', 'alpha', 'beta', 'gamma']
    results = {}

    fig, axs = plt.subplots(2, 2, figsize=(15, 10), sharex=True, sharey=True)
    bar_width = 0.25
    for i, frequency_band in enumerate(frequency_bands):
        subject_ratios_1 = []
        subject_ratios_2 = []
        subject_ratios_3 = []
        subject_agreement_12 = []
        subject_agreement_13 = []
        subject_agreement_23 = []

        for subject_index in cfg.dataset.test_subject_indices:
            common_channels_1 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
            common_channels_2 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
            common_channels_3 = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

            num_common_channels_1 = len(common_channels_1)
            num_common_channels_2 = len(common_channels_2)
            num_common_channels_3 = len(common_channels_3)

            num_significant_channels_1 = len(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
            num_significant_channels_2 = len(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
            num_significant_channels_3 = len(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

            denominator_1 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_1))
            denominator_2 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_2))
            denominator_3 = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), num_significant_channels_3))

            ratio_1 = num_common_channels_1 / denominator_1
            ratio_2 = num_common_channels_2 / denominator_2
            ratio_3 = num_common_channels_3 / denominator_3

            subject_ratios_1.append(ratio_1)
            subject_ratios_2.append(ratio_2)
            subject_ratios_3.append(ratio_3)

            top_10_set_1 = set(all_subjects_highest_corrs_1[subject_index][frequency_band].keys())
            top_10_set_2 = set(all_subjects_highest_corrs_2[subject_index][frequency_band].keys())
            top_10_set_3 = set(all_subjects_highest_corrs_3[subject_index][frequency_band].keys())

            agreement_12 = len(top_10_set_1 & top_10_set_2) / 10
            agreement_13 = len(top_10_set_1 & top_10_set_3) / 10
            agreement_23 = len(top_10_set_2 & top_10_set_3) / 10

            subject_agreement_12.append(agreement_12)
            subject_agreement_13.append(agreement_13)
            subject_agreement_23.append(agreement_23)

        mean_ratio_1 = np.mean(subject_ratios_1)
        mean_ratio_2 = np.mean(subject_ratios_2)
        mean_ratio_3 = np.mean(subject_ratios_3)
        std_ratio_1 = np.std(subject_ratios_1)
        std_ratio_2 = np.std(subject_ratios_2)
        std_ratio_3 = np.std(subject_ratios_3)
        mean_agreement_12 = np.mean(subject_agreement_12)
        mean_agreement_13 = np.mean(subject_agreement_13)
        mean_agreement_23 = np.mean(subject_agreement_23)
        std_agreements_12 = np.std(subject_agreement_12)
        std_agreements_13 = np.std(subject_agreement_13)
        std_agreements_23 = np.std(subject_agreement_23)

        results[frequency_band] = {
            'mean_ratio_1': mean_ratio_1,
            'mean_ratio_2': mean_ratio_2,
            'mean_ratio_3': mean_ratio_3,
            'std_ratio_1': std_ratio_1,
            'std_ratio_2': std_ratio_2,
            'std_ratio_3': std_ratio_3,
            'mean_agreement_12': mean_agreement_12,
            'mean_agreement_13': mean_agreement_13,
            'mean_agreement_23': mean_agreement_23,
            'std_agreement_12': std_agreements_12,
            'std_agreement_13': std_agreements_13,
            'std_agreement_23': std_agreements_23
        }



    
        ax = axs[i // 2, i % 2]
        index = np.arange(len(cfg.dataset.test_subject_indices))

        bar1 = ax.bar(index, subject_ratios_1, bar_width, label='absolute power')
        bar2 = ax.bar(index + bar_width, subject_ratios_2, bar_width, label='relative power')
        bar3 = ax.bar(index + 2 * bar_width, subject_ratios_3, bar_width, label='periodic component power')
        ax.axhline(y=0.3, color='red', linestyle='--', linewidth=2)
        ax.set_xlabel('subject index', fontsize=14)
        ax.set_ylabel('ratio of common channels', fontsize=14)
        ax.set_title(f'common channels ratio {frequency_band} band', fontsize=16)
        ax.set_xticks(index + bar_width)
        ax.set_xticklabels(np.arange(len(cfg.dataset.test_subject_indices)), rotation=45, fontsize=12)
        ax.tick_params(axis='y', labelsize=12)
        if i == 1:
            ax.legend(fontsize=12)

    plt.tight_layout()
    plt.savefig("common_channels_ratio_all_bands_all_features.png")
    plt.show()

    # Plot mean, std, and agreement in a single figure with two subplots
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    # Subplot 1: Mean and Std of Common Channels Ratio Across Frequency Bands
    index = np.arange(len(frequency_bands))
    bar_width = 0.3
    mean_ratios_1 = [results[band]['mean_ratio_1'] for band in frequency_bands]
    mean_ratios_2 = [results[band]['mean_ratio_2'] for band in frequency_bands]
    mean_ratios_3 = [results[band]['mean_ratio_3'] for band in frequency_bands]
    std_ratios_1 = [results[band]['std_ratio_1'] for band in frequency_bands]
    std_ratios_2 = [results[band]['std_ratio_2'] for band in frequency_bands]
    std_ratios_3 = [results[band]['std_ratio_3'] for band in frequency_bands]

    ax = axs[0]
    bar1 = ax.bar(index, mean_ratios_1, bar_width, yerr=std_ratios_1, label='absolute power')
    bar2 = ax.bar(index + bar_width, mean_ratios_2, bar_width, yerr=std_ratios_2, label='relative power')
    bar3 = ax.bar(index + 2 * bar_width, mean_ratios_3, bar_width, yerr=std_ratios_3, label='periodic component power')

    ax.set_xlabel('frequency Band', fontsize=14)
    ax.set_ylabel('mean ratio of common channels', fontsize=14)
    ax.set_title('mean and std of common channels ratio', fontsize=13)
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(frequency_bands, fontsize=12)
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(fontsize=12, loc='upper left')

    # Subplot 2: Mean Agreement Between Dictionaries Across Frequency Bands
    mean_agreements_12 = [results[band]['mean_agreement_12'] for band in frequency_bands]
    mean_agreements_13 = [results[band]['mean_agreement_13'] for band in frequency_bands]
    mean_agreements_23 = [results[band]['mean_agreement_23'] for band in frequency_bands]
    std_agreements_12 = [results[band].get('std_agreement_12', 0) for band in frequency_bands]
    std_agreements_13 = [results[band].get('std_agreement_13', 0) for band in frequency_bands]
    std_agreements_23 = [results[band].get('std_agreement_23', 0) for band in frequency_bands]

    ax = axs[1]
    bar1 = ax.bar(index, mean_agreements_12, bar_width, yerr=std_agreements_12, label='absolute-relative power', color='brown')
    bar2 = ax.bar(index + bar_width, mean_agreements_13, bar_width, yerr=std_agreements_13, label='absolute-periodic power', color='cyan')
    bar3 = ax.bar(index + 2 * bar_width, mean_agreements_23, bar_width, yerr=std_agreements_23, label='relative-periodic power', color='magenta')

    ax.set_xlabel('frequency band', fontsize=14)
    ax.set_ylabel('mean agreement ratio', fontsize=14)
    ax.set_title('mean agreement between power features', fontsize=13)
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(frequency_bands, fontsize=12)
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(fontsize=12, loc="upper left")

    plt.tight_layout()
    plt.savefig("combined_common_channels_ratio_and_agreement_all_features.png")
    plt.show()

    return results

# Example usage:
top_10_absolute_power = np.load("top_10_absolute_power.npy", allow_pickle=True).item()
top_10_relative_power = np.load("top_10_relative_power.npy", allow_pickle=True).item()
top_10_periodic_power = np.load("top_10_periodic_component.npy", allow_pickle=True).item()
results = compare_channels_all_bands(cfg, top_10_per_subject, top_10_absolute_power, top_10_relative_power, top_10_periodic_power)

In [ ]:
results 

# rank correlations for top 60 channels

compute absolute correlation for each channel and compute rank correlation with top-60 of explanatio function

In [ ]:
#corrs_per_freqband = plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=2, treshhold=0, show_plot=False)
corrs_per_freqband = {}
for subject_index in cfg.dataset.test_subject_indices:
    corrs_per_freqband[subject_index] = plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index, treshhold=-1, show_plot=False)

In [ ]:
corrs_per_freqband

In [ ]:
#np.save("highest_corrs_absolute_power.npy", corrs_per_freqband)

In [ ]:
corrs_per_freqband_abs = {}
for subject_index in cfg.dataset.test_subject_indices:
    corrs_per_freqband_abs[subject_index] = plot_power_amplitude(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index,show_plot=False)[2]

In [ ]:
corrs_per_freqband_abs

In [ ]:
np.save("highest_corrs_absolute_power_abs.npy", corrs_per_freqband)

In [ ]:
top_60_per_subject_dict = {subject_index: top_60_per_subject_dict[i] for i, subject_index in enumerate(cfg.dataset.test_subject_indices)}

In [ ]:
top_60_per_subject_dict[1].values()

In [ ]:
from scipy.stats import spearmanr
rank_correlations = {}

for freq_band in freq_bands.keys():
    rank_correlations[freq_band] = {}
    for subject_index in cfg.dataset.test_subject_indices:
        top_60_channels = top_60_per_subject_dict[subject_index]
        corrs = corrs_per_freqband[subject_index][freq_band]
        print(subject_index)

        # Compute the Spearman rank correlation
        
        rank_corr, pval = spearmanr(list(top_60_channels.values()), list(corrs.values()))
        print(f"Rank correlation: {rank_corr:.2f}, p-value: {pval:.2f}")
      
        
        rank_correlations[freq_band][subject_index] = (rank_corr, pval)




In [ ]:
freq_bands

In [ ]:
import matplotlib.pyplot as plt


# Plot rank correlations for each frequency band
for freq_band in freq_bands:
    plt.figure(figsize=(12, 6))
    subjects = list(rank_correlations[freq_band].keys())
    rank_corrs = [rank_correlations[freq_band][subject][0] for subject in subjects]
    p_values = [rank_correlations[freq_band][subject][1] for subject in subjects]

    plt.bar(subjects, rank_corrs, color='blue', alpha=0.7, label='Rank Correlation')
    plt.scatter(subjects, rank_corrs, c=['red' if p < (0.05/60) else 'black' for p in p_values], label='p < 0.05', zorder=5)
    
    plt.xlabel('Subject Index')
    plt.ylabel('Rank Correlation')
    plt.title(f'Rank Correlation for {freq_band} Band')
    plt.axhline(y=0, color='gray', linestyle='--')
    plt.legend()
    plt.show()